In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

RESULTS_DIR = Path("/yunity/tkerby2/Local_CorEx/Local_CorEx_Experiments/ablation_data_v2")

gaussian_df = pd.read_csv(RESULTS_DIR / "gaussian_ablation.csv")
df = gaussian_df

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

metrics = [
    "ave_group_cosine",
    "ave_auc_group_prc",
    "ave_lf_cosine",
    "ave_auc_lf_prc",
]

gaussian_df["setup"] = np.where(gaussian_df["disjoint"], "Disjoint", "Non-disjoint")

def method_label(row):
    if row["partition_method"] == "global":
        return "Linear" if row["model_kind"] == "linear" else "Bio"
    return "Local Lin" if row["model_kind"] == "linear" else "Local Bio"

gaussian_df["method"] = gaussian_df.apply(method_label, axis=1)

paper_metrics = {
    "Group": {
        "Cosine Dist": "ave_group_cosine",
        "AUCPRC": "ave_auc_group_prc",
    },
    "Top K Latent Factor": {
        "Cosine Dist": "ave_lf_cosine",
        "AUCPRC": "ave_auc_lf_prc",
    },
}

def build_paper_table(dataframe, alpha_values=(0.0, 1.0)):
    base = (
        dataframe
        .query("dataset == 'gaussian'")
        .query("n_noise == 0")
        .query("alpha in @alpha_values")
    )

    pieces = []
    for metric_group, submetrics in paper_metrics.items():
        for metric_label, source_col in submetrics.items():
            part = (
                base
                .groupby(["setup", "cluster_size", "method", "alpha"])[source_col]
                .mean()
                .unstack("alpha")
                .reindex(columns=list(alpha_values))
                .rename(columns={0.0: "α = 0.0", 1.0: "α = 1.0"})
                .round(3)
            )
            part.columns = pd.MultiIndex.from_tuples(
                [
                    (metric_group, metric_label, alpha_label)
                    for alpha_label in part.columns
                ],
                names=["metric_group", "metric", "alpha"],
            )
            pieces.append(part)

    table = pd.concat(pieces, axis=1)
    return table.sort_index(level=["setup", "cluster_size", "method"])

paper_table = build_paper_table(gaussian_df)

In [23]:
gaussian_df.head(10)

,dataset,disjoint,rep,cluster_size,alpha,n_noise,partition_method,model_kind,n_hidden,corex_time,total_time,ave_group_cosine,ave_auc_group_prc,ave_lf_cosine,ave_auc_lf_prc,partition_ari,partition_time,cluster_purity,setup,method
0,gaussian,False,0,50,0.0,0,global,linear,5,0.043388,0.043388,0.170175,0.921127,0.131554,0.920609,NaN,NaN,NaN,Non-disjoint,Linear
1,gaussian,False,0,50,0.0,0,global,bio,5,1.418789,1.418789,0.261747,0.838823,0.064026,1.000000,NaN,NaN,NaN,Non-disjoint,Bio
2,gaussian,False,0,50,0.0,0,oracle,linear,5,0.099645,0.099658,0.152373,0.927335,0.171433,0.909169,1.000000,0.000013,1.00,Non-disjoint,Local Lin
3,gaussian,False,0,50,0.0,0,oracle,bio,5,3.227081,3.227095,0.211037,0.836583,0.152849,0.936744,1.000000,0.000013,1.00,Non-disjoint,Local Bio
4,gaussian,False,0,50,0.0,0,phate,linear,5,0.099396,0.317679,0.171623,0.885966,0.186484,0.868305,0.001543,0.218283,0.46,Non-disjoint,Local Lin
5,gaussian,False,0,50,0.0,0,phate,bio,5,2.555051,2.773334,0.219107,0.809759,0.168166,0.895266,0.001543,0.218283,0.46,Non-disjoint,Local Bio
6,gaussian,False,0,50,0.0,0,raw_kmeans,linear,5,0.108334,0.110462,0.214148,0.872807,0.229045,0.863675,-0.009302,0.002128,0.51,Non-disjoint,Local Lin
7,gaussian,False,0,50,0.0,0,raw_kmeans,bio,5,2.468758,2.470887,0.297810,0.767381,0.199852,0.882395,-0.009302,0.002128,0.51,Non-disjoint,Local Bio
8,gaussian,False,0,50,0.0,0,random,linear,5,0.093590,0.093672,0.176944,0.902136,0.188678,0.885071,-0.003591,0.000082,0.54,Non-disjoint,Local Lin
9,gaussian,False,0,50,0.0,0,random,bio,5,2.719421,2.719503,0.295238,0.761001,0.228308,0.838495,-0.003591,0.000082,0.54,Non-disjoint,Local Bio


In [24]:
# Gaussian Comparison to the Paper Table
#
# The paper-style table below mirrors the old layout with row groups for
# `Disjoint` and `Non-disjoint` and column groups for `Group` vs
# `Top K Latent Factor`.
# The current experiments use `alpha in {0.0, 0.5, 1.0}`, but the paper table
# comparison here focuses on `alpha = 0.0` and `alpha = 1.0` to match the
# reference image.

In [25]:
paper_table

metric_group                              Group                         Top K Latent Factor                        
metric                              Cosine Dist          AUCPRC                 Cosine Dist          AUCPRC        
alpha                                   α = 0.0 α = 1.0 α = 0.0 α = 1.0             α = 0.0 α = 1.0 α = 0.0 α = 1.0
setup        cluster_size method                                                                                   
Disjoint     50           Bio             0.442   0.479   0.608   0.527               0.264   0.313   0.829   0.851
                          Linear          0.491   0.415   0.605   0.583               0.296   0.221   0.863   0.809
                          Local Bio       0.398   0.292   0.640   0.756               0.361   0.227   0.679   0.826
                          Local Lin       0.385   0.195   0.706   0.869               0.359   0.196   0.729   0.870
             500          Bio             0.387   0.458   0.703   0.557               0.138   0.304   0.985   0.850
                          Linear          0.493   0.396   0.621   0.620               0.273   0.211   0.864   0.866
                          Local Bio       0.327   0.162   0.743   0.859               0.259   0.136   0.816   0.897
                          Local Lin       0.374   0.125   0.718   0.905               0.316   0.127   0.768   0.910
             5000         Bio             0.402   0.462   0.671   0.548               0.185   0.310   0.949   0.812
                          Linear          0.513   0.388   0.612   0.628               0.275   0.184   0.882   0.888
                          Local Bio       0.237   0.160   0.810   0.852               0.183   0.128   0.873   0.900
                          Local Lin       0.281   0.113   0.800   0.912               0.236   0.114   0.836   0.916
Non-disjoint 50           Bio             0.224   0.279   0.857   0.753               0.092   0.234   0.956   0.857
                          Linear          0.127   0.112   0.966   0.963               0.080   0.096   0.977   0.979
                          Local Bio       0.245   0.213   0.818   0.833               0.177   0.175   0.896   0.897
                          Local Lin       0.168   0.142   0.922   0.953               0.181   0.153   0.912   0.946
             500          Bio             0.115   0.246   0.951   0.793               0.066   0.228   0.985   0.869
                          Linear          0.089   0.084   1.000   0.962               0.035   0.058   1.000   0.980
                          Local Bio       0.138   0.129   0.931   0.922               0.078   0.084   0.986   0.980
                          Local Lin       0.085   0.064   0.995   0.990               0.082   0.063   0.988   0.992
             5000         Bio             0.071   0.253   1.000   0.757               0.025   0.184   1.000   0.887
                          Linear          0.084   0.082   1.000   0.961               0.028   0.055   1.000   0.979
                          Local Bio       0.134   0.126   0.935   0.915               0.079   0.092   0.986   0.959
                          Local Lin       0.077   0.058   0.994   0.990               0.072   0.056   0.988   0.992

In [26]:
paper_alpha_delta = (
    paper_table
    .xs("α = 1.0", level="alpha", axis=1)
    - paper_table.xs("α = 0.0", level="alpha", axis=1)
).round(3)

paper_alpha_delta

metric_group                              Group        Top K Latent Factor       
metric                              Cosine Dist AUCPRC         Cosine Dist AUCPRC
setup        cluster_size method                                                 
Disjoint     50           Bio             0.037 -0.081               0.049  0.022
                          Linear         -0.076 -0.022              -0.075 -0.054
                          Local Bio      -0.106  0.116              -0.134  0.147
                          Local Lin      -0.190  0.163              -0.163  0.141
             500          Bio             0.071 -0.146               0.166 -0.135
                          Linear         -0.097 -0.001              -0.062  0.002
                          Local Bio      -0.165  0.116              -0.123  0.081
                          Local Lin      -0.249  0.187              -0.189  0.142
             5000         Bio             0.060 -0.123               0.125 -0.137
                          Linear         -0.125  0.016              -0.091  0.006
                          Local Bio      -0.077  0.042              -0.055  0.027
                          Local Lin      -0.168  0.112              -0.122  0.080
Non-disjoint 50           Bio             0.055 -0.104               0.142 -0.099
                          Linear         -0.015 -0.003               0.016  0.002
                          Local Bio      -0.032  0.015              -0.002  0.001
                          Local Lin      -0.026  0.031              -0.028  0.034
             500          Bio             0.131 -0.158               0.162 -0.116
                          Linear         -0.005 -0.038               0.023 -0.020
                          Local Bio      -0.009 -0.009               0.006 -0.006
                          Local Lin      -0.021 -0.005              -0.019  0.004
             5000         Bio             0.182 -0.243               0.159 -0.113
                          Linear         -0.002 -0.039               0.027 -0.021
                          Local Bio      -0.008 -0.020               0.013 -0.027
                          Local Lin      -0.019 -0.004              -0.016  0.004

In [27]:
counts = (
    gaussian_df
    .query("dataset == 'gaussian'")
    .query("n_noise == 0")
    .query("alpha in [0.0, 1.0]")
    .groupby(["setup", "cluster_size", "method", "alpha"])
    .size()
    .unstack("alpha")
    .reindex(columns=[0.0, 1.0])
    .rename(columns={0.0: "α = 0.0", 1.0: "α = 1.0"})
    .fillna(0)
    .astype(int)
)

counts

alpha                                α = 0.0  α = 1.0
setup        cluster_size method                     
Disjoint     50           Bio              4        4
                          Linear           4        4
                          Local Bio       16       16
                          Local Lin       16       16
             500          Bio              4        4
                          Linear           4        4
                          Local Bio       16       16
                          Local Lin       16       16
             5000         Bio              4        4
                          Linear           4        4
                          Local Bio       16       16
                          Local Lin       16       16
Non-disjoint 50           Bio              4        4
                          Linear           4        4
                          Local Bio       16       16
                          Local Lin       16       16
             500          Bio              4        4
                          Linear           4        4
                          Local Bio       16       16
                          Local Lin       16       16
             5000         Bio              4        4
                          Linear           4        4
                          Local Bio       16       16
                          Local Lin       16       16

In [28]:
fixed_factors = (
    df
    .query("dataset == 'gaussian'")
    .query("n_noise == 0")
    .query("n_hidden == 5")
    .query("alpha in [0.0, 0.5, 1.0]")
    .query("partition_method in ['global', 'oracle', 'phate', 'raw_kmeans', 'random']")
    .pivot_table(
        index=["setup", "cluster_size", "model_kind", "partition_method"],
        columns="alpha",
        values=metrics,
        aggfunc="mean",
    )
    .round(3)
 )

fixed_factors.columns = [f"{metric}_alpha_{alpha:g}" for metric, alpha in fixed_factors.columns]
fixed_factors = fixed_factors.reset_index()
fixed_factors

,setup,cluster_size,model_kind,partition_method,ave_auc_group_prc_alpha_0,ave_auc_group_prc_alpha_0.5,ave_auc_group_prc_alpha_1,ave_auc_lf_prc_alpha_0,ave_auc_lf_prc_alpha_0.5,ave_auc_lf_prc_alpha_1,ave_group_cosine_alpha_0,ave_group_cosine_alpha_0.5,ave_group_cosine_alpha_1,ave_lf_cosine_alpha_0,ave_lf_cosine_alpha_0.5,ave_lf_cosine_alpha_1
0,Disjoint,50,bio,global,0.608,0.577,0.527,0.829,0.783,0.851,0.442,0.429,0.479,0.264,0.265,0.313
1,Disjoint,50,bio,oracle,0.872,0.828,0.865,0.898,0.881,0.898,0.179,0.236,0.189,0.152,0.180,0.158
2,Disjoint,50,bio,phate,0.564,0.816,0.846,0.614,0.899,0.938,0.471,0.247,0.228,0.425,0.170,0.135
3,Disjoint,50,bio,random,0.536,0.538,0.511,0.565,0.592,0.569,0.503,0.467,0.487,0.462,0.416,0.452
4,Disjoint,50,bio,raw_kmeans,0.589,0.826,0.801,0.638,0.887,0.899,0.441,0.232,0.266,0.405,0.177,0.163
5,Disjoint,50,linear,global,0.605,0.567,0.583,0.863,0.778,0.809,0.491,0.464,0.415,0.296,0.276,0.221
6,Disjoint,50,linear,oracle,0.969,0.963,0.962,0.966,0.957,0.960,0.128,0.133,0.114,0.134,0.141,0.120
7,Disjoint,50,linear,phate,0.618,0.954,0.962,0.652,0.948,0.960,0.482,0.140,0.114,0.437,0.148,0.120
8,Disjoint,50,linear,random,0.576,0.588,0.588,0.614,0.612,0.597,0.487,0.452,0.438,0.449,0.439,0.426
9,Disjoint,50,linear,raw_kmeans,0.661,0.936,0.962,0.685,0.933,0.960,0.443,0.150,0.114,0.414,0.155,0.120


In [29]:
avg_over_factors = (
    df
    .query("dataset == 'gaussian'")
    .query("n_noise == 0")
    .query("alpha in [0.0, 0.5, 1.0]")
    .query("partition_method in ['global', 'oracle', 'phate', 'raw_kmeans', 'random']")
    .groupby(["setup", "cluster_size", "model_kind", "partition_method", "alpha"])[metrics]
    .mean()
    .round(3)
    .reset_index()
 )

avg_over_factors

,setup,cluster_size,model_kind,partition_method,alpha,ave_group_cosine,ave_auc_group_prc,ave_lf_cosine,ave_auc_lf_prc
0,Disjoint,50,bio,global,0.0,0.442,0.608,0.264,0.829
1,Disjoint,50,bio,global,0.5,0.429,0.577,0.265,0.783
2,Disjoint,50,bio,global,1.0,0.479,0.527,0.313,0.851
3,Disjoint,50,bio,oracle,0.0,0.179,0.872,0.152,0.898
4,Disjoint,50,bio,oracle,0.5,0.236,0.828,0.180,0.881
5,Disjoint,50,bio,oracle,1.0,0.189,0.865,0.158,0.898
6,Disjoint,50,bio,phate,0.0,0.471,0.564,0.425,0.614
7,Disjoint,50,bio,phate,0.5,0.247,0.816,0.170,0.899
8,Disjoint,50,bio,phate,1.0,0.228,0.846,0.135,0.938
9,Disjoint,50,bio,random,0.0,0.503,0.536,0.462,0.565


In [30]:
partition_compare = (
    df
    .query("dataset == 'gaussian'")
    .query("n_noise == 0")
    .query("model_kind == 'linear'")
    .query("n_hidden == 5")
    .query("alpha in [0.0, 0.5, 1.0]")
    .groupby(["setup", "cluster_size", "alpha", "partition_method"])[
        [
            "ave_group_cosine",
            "ave_auc_group_prc",
            "ave_lf_cosine",
            "ave_auc_lf_prc",
            "partition_ari",
            "cluster_purity",
            "corex_time",
            "partition_time",
            "total_time",
        ]
    ]
    .mean()
    .round(3)
 )

partition_compare

ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc  partition_ari  cluster_purity  corex_time  partition_time  total_time
setup        cluster_size alpha partition_method                                                                                                                                           
Disjoint     50           0.0   global                       0.491              0.605          0.296           0.863            NaN             NaN       0.064             NaN       0.064
                                oracle                       0.128              0.969          0.134           0.966          1.000           1.000       0.123           0.000       0.123
                                phate                        0.482              0.618          0.437           0.652          0.003           0.528       0.133           0.282       0.415
                                random                       0.487              0.576          0.449           0.614         -0.006           0.502       0.131           0.000       0.131
                                raw_kmeans                   0.443              0.661          0.414           0.685          0.002           0.535       0.156           0.014       0.169
                          0.5   global                       0.464              0.567          0.276           0.778            NaN             NaN       0.109             NaN       0.109
                                oracle                       0.133              0.963          0.141           0.957          1.000           1.000       0.111           0.000       0.111
                                phate                        0.140              0.954          0.148           0.948          0.980           0.995       0.108           0.204       0.313
                                random                       0.452              0.588          0.439           0.612         -0.006           0.502       0.160           0.000       0.160
                                raw_kmeans                   0.150              0.936          0.155           0.933          0.931           0.982       0.082           0.013       0.095
                          1.0   global                       0.415              0.583          0.221           0.809            NaN             NaN       0.110             NaN       0.110
                                oracle                       0.114              0.962          0.120           0.960          1.000           1.000       0.079           0.000       0.079
                                phate                        0.114              0.962          0.120           0.960          1.000           1.000       0.070           0.279       0.349
                                random                       0.438              0.588          0.426           0.597         -0.006           0.502       0.238           0.000       0.238
                                raw_kmeans                   0.114              0.962          0.120           0.960          1.000           1.000       0.071           0.015       0.086
             500          0.0   global                       0.493              0.621          0.273           0.864            NaN             NaN       0.095             NaN       0.095
                                oracle                       0.035              1.000          0.034           1.000          1.000           1.000       0.046           0.000       0.046
                                phate                        0.468              0.653          0.389           0.716          0.004           0.530       0.124           2.430       2.555
                                random                       0.497              0.615          0.420           0.691         -0.000           0.511       0.119           0.000       0.119
                                raw_kmeans                   0.494              0.602          0.420  

In [31]:
table1 = (
    df
    .query("dataset == 'gaussian'")
    .query("n_noise == 0")
    .query("n_hidden == 5")
    .query("alpha in [0.0, 0.5, 1.0]")
    .query("partition_method in ['global', 'oracle', 'phate', 'raw_kmeans', 'random']")
    .pivot_table(
        index=["setup", "cluster_size", "model_kind", "partition_method"],
        columns="alpha",
        values=metrics,
        aggfunc="mean",
    )
    .round(3)
 )

table1.columns = [f"{metric}, alpha={alpha:g}" for metric, alpha in table1.columns]
table1 = table1.reset_index()

table1

,setup,cluster_size,model_kind,partition_method,"ave_auc_group_prc, alpha=0","ave_auc_group_prc, alpha=0.5","ave_auc_group_prc, alpha=1","ave_auc_lf_prc, alpha=0","ave_auc_lf_prc, alpha=0.5","ave_auc_lf_prc, alpha=1","ave_group_cosine, alpha=0","ave_group_cosine, alpha=0.5","ave_group_cosine, alpha=1","ave_lf_cosine, alpha=0","ave_lf_cosine, alpha=0.5","ave_lf_cosine, alpha=1"
0,Disjoint,50,bio,global,0.608,0.577,0.527,0.829,0.783,0.851,0.442,0.429,0.479,0.264,0.265,0.313
1,Disjoint,50,bio,oracle,0.872,0.828,0.865,0.898,0.881,0.898,0.179,0.236,0.189,0.152,0.180,0.158
2,Disjoint,50,bio,phate,0.564,0.816,0.846,0.614,0.899,0.938,0.471,0.247,0.228,0.425,0.170,0.135
3,Disjoint,50,bio,random,0.536,0.538,0.511,0.565,0.592,0.569,0.503,0.467,0.487,0.462,0.416,0.452
4,Disjoint,50,bio,raw_kmeans,0.589,0.826,0.801,0.638,0.887,0.899,0.441,0.232,0.266,0.405,0.177,0.163
5,Disjoint,50,linear,global,0.605,0.567,0.583,0.863,0.778,0.809,0.491,0.464,0.415,0.296,0.276,0.221
6,Disjoint,50,linear,oracle,0.969,0.963,0.962,0.966,0.957,0.960,0.128,0.133,0.114,0.134,0.141,0.120
7,Disjoint,50,linear,phate,0.618,0.954,0.962,0.652,0.948,0.960,0.482,0.140,0.114,0.437,0.148,0.120
8,Disjoint,50,linear,random,0.576,0.588,0.588,0.614,0.612,0.597,0.487,0.452,0.438,0.449,0.439,0.426
9,Disjoint,50,linear,raw_kmeans,0.661,0.936,0.962,0.685,0.933,0.960,0.443,0.150,0.114,0.414,0.155,0.120


# Nonlinear Results

In [43]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/yunity/tkerby2/Local_CorEx/Local_CorEx_Experiments/ablation_data_v2/nonlinear_ablation.csv",
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

df["setup"] = "Nonlinear"

def method_label_nonlinear(row):
    if row["partition_method"] == "global":
        return "Linear" if row["model_kind"] == "linear" else "Bio"
    return "Local Lin" if row["model_kind"] == "linear" else "Local Bio"

df["method"] = df.apply(method_label_nonlinear, axis=1)

noise_levels = [0, 25, 75]

def build_nonlinear_paper_table(dataframe, noise_values=noise_levels):
    base = dataframe.query("dataset == 'nonlinear'")

    pieces = []
    for metric_group, submetrics in paper_metrics.items():
        for metric_label, source_col in submetrics.items():
            part = (
                base
                .groupby(["setup", "cluster_size", "method", "n_noise"])[source_col]
                .mean()
                .unstack("n_noise")
                .reindex(columns=list(noise_values))
                .rename(columns={0: "n_noise = 0", 25: "n_noise = 25", 75: "n_noise = 75"})
                .round(3)
            )
            part.columns = pd.MultiIndex.from_tuples(
                [
                    (metric_group, metric_label, noise_label)
                    for noise_label in part.columns
                ],
                names=["metric_group", "metric", "n_noise"],
            )
            pieces.append(part)

    table = pd.concat(pieces, axis=1)
    return table.sort_index(level=["setup", "cluster_size", "method"])

nonlinear_paper_table = build_nonlinear_paper_table(df)

In [44]:
metrics = [
    "ave_group_cosine",
    "ave_auc_group_prc",
    "ave_lf_cosine",
    "ave_auc_lf_prc",
]

In [45]:
nonlinear_paper_table

metric_group                           Group                                                                 Top K Latent Factor                                                                
metric                           Cosine Dist                                AUCPRC                                   Cosine Dist                                AUCPRC                          
n_noise                          n_noise = 0 n_noise = 25 n_noise = 75 n_noise = 0 n_noise = 25 n_noise = 75         n_noise = 0 n_noise = 25 n_noise = 75 n_noise = 0 n_noise = 25 n_noise = 75
setup     cluster_size method                                                                                                                                                                   
Nonlinear 50           Bio             0.598        0.621        0.782       0.470        0.328        0.159               0.472        0.549        0.766       0.573        0.385        0.181
                       Linear          0.548        0.678        0.769       0.448        0.266        0.157               0.464        0.646        0.748       0.538        0.298        0.178
                       Local Bio       0.573        0.651        0.771       0.455        0.297        0.160               0.650        0.737        0.833       0.384        0.232        0.117
                       Local Lin       0.526        0.675        0.751       0.471        0.244        0.166               0.649        0.754        0.815       0.379        0.205        0.125
          500          Bio             0.575        0.595        0.705       0.629        0.460        0.282               0.439        0.534        0.657       0.738        0.530        0.329
                       Linear          0.548        0.599        0.687       0.598        0.527        0.242               0.433        0.513        0.636       0.761        0.649        0.293
                       Local Bio       0.562        0.612        0.667       0.563        0.417        0.309               0.645        0.709        0.767       0.466        0.315        0.214
                       Local Lin       0.531        0.623        0.724       0.560        0.390        0.255               0.631        0.716        0.776       0.455        0.288        0.182
          5000         Bio             0.523        0.666        0.699       0.673        0.498        0.409               0.439        0.579        0.636       0.778        0.597        0.483
                       Linear          0.663        0.654        0.636       0.585        0.559        0.548               0.407        0.497        0.567       0.834        0.741        0.650
                       Local Bio       0.548        0.572        0.640       0.639        0.563        0.443               0.692        0.713        0.774       0.467        0.401        0.290
                       Local Lin       0.569        0.614        0.664       0.652        0.578        0.465               0.648        0.689        0.753       0.531        0.451        0.337

# Nonlinear Comparison to the Paper Table

This table mirrors the Gaussian paper-style layout, but uses `n_noise` as the swept condition so we can check whether the same pattern holds under increasing noise.

In [34]:
summary = (
    df
    .query("dataset == 'nonlinear'")
    .query("partition_method in ['global', 'oracle', 'phate', 'raw_kmeans', 'random']")
    .groupby(["setup", "cluster_size", "model_kind", "partition_method", "n_noise"])[metrics]
    .mean(numeric_only=True)
    .round(3)
 )

summary = (
    summary
    .reset_index()
    .sort_values(
        ["setup", "cluster_size", "model_kind", "partition_method", "n_noise"],
        ascending=[True, True, True, True, True],
    )
    .set_index(["setup", "cluster_size", "model_kind", "partition_method", "n_noise"])
)

summary

ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc
setup     cluster_size model_kind partition_method n_noise                                                                    
Nonlinear 50           bio        global           0                   0.598              0.470          0.472           0.573
                                                   25                  0.621              0.328          0.549           0.385
                                                   75                  0.782              0.159          0.766           0.181
                                  oracle           0                   0.416              0.624          0.525           0.514
                                                   25                  0.612              0.336          0.725           0.247
                                                   75                  0.771              0.163          0.833           0.119
                                  phate            0                   0.620              0.406          0.669           0.350
                                                   25                  0.657              0.270          0.744           0.219
                                                   75                  0.739              0.210          0.800           0.152
                                  random           0                   0.666              0.383          0.746           0.316
                                                   25                  0.681              0.290          0.760           0.225
                                                   75                  0.814              0.101          0.862           0.081
                                  raw_kmeans       0                   0.590              0.408          0.659           0.357
                                                   25                  0.653              0.290          0.721           0.239
                                                   75                  0.761              0.167          0.838           0.117
                       linear     global           0                   0.548              0.448          0.464           0.538
                                                   25                  0.678              0.266          0.646           0.298
                                                   75                  0.769              0.157          0.748           0.178
                                  oracle           0                   0.498              0.545          0.636           0.425
                                                   25                  0.691              0.255          0.772           0.213
                                                   75                  0.678              0.252          0.770           0.174
                                  phate            0                   0.544              0.426          0.658           0.349
                                                   25                  0.688              0.234          0.756           0.198
                                                   75                  0.769              0.136          0.828           0.109
                                  random           0                   0.557              0.422          0.665           0.350
                                                   25                  0.656              0.248          0.754           0.202
                                                   75                  0.763              0.155          0.825           0.118
                                  raw_kmeans       0                   0.503              0.491          0.637           0.393
                                                   25                  0.665              0.240          0.733           0.207
                                                   75                  0.794              0.122          0.839           0.

In [35]:
summary = (
    df
    .query("dataset == 'nonlinear'")
    .groupby(["cluster_size", "partition_method"])[[
        "ave_group_cosine",
        "ave_auc_group_prc",
        "ave_lf_cosine",
        "ave_auc_lf_prc",
        "corex_time",
        "partition_time",
        "total_time",
    ]]
    .mean(numeric_only=True)
    .round(3)
    .sort_values(
        ["cluster_size", "ave_auc_group_prc"],
        ascending=[True, False],
    )
)

summary

ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc  corex_time  partition_time  total_time
cluster_size partition_method                                                                                                            
50           oracle                       0.611              0.362          0.710           0.282       3.296           0.000       3.296
             global                       0.666              0.305          0.607           0.359       2.340             NaN       2.340
             raw_kmeans                   0.661              0.286          0.738           0.235       3.264           0.014       3.277
             phate                        0.669              0.280          0.743           0.230       3.139           0.171       3.310
             random                       0.689              0.267          0.769           0.215       3.324           0.000       3.324
500          oracle                       0.491              0.644          0.597           0.470      13.929           0.000      13.929
             global                       0.618              0.456          0.535           0.550      13.559             NaN      13.559
             random                       0.649              0.365          0.734           0.282      14.019           0.000      14.019
             phate                        0.662              0.327          0.751           0.253      13.034           2.157      15.191
             raw_kmeans                   0.676              0.326          0.748           0.275      13.930           0.034      13.965
5000         oracle                       0.504              0.627          0.611           0.476     127.762           0.000     127.763
             random                       0.609              0.580          0.739           0.406     136.100           0.000     136.100
             global                       0.640              0.545          0.521           0.681     133.089             NaN     133.089
             raw_kmeans                   0.647              0.543          0.717           0.443     126.409           0.066     126.475
             phate                        0.645              0.476          0.779           0.326     126.644           8.328     134.972

In [36]:
wide = (
    df
    .query("dataset == 'nonlinear'")
    .query("partition_method in ['global', 'oracle', 'phate', 'raw_kmeans', 'random']")
    .pivot_table(
        index=["setup", "cluster_size", "model_kind", "partition_method"],
        columns="n_noise",
        values=metrics,
        aggfunc="mean",
    )
    .round(3)
 )

wide.columns = [f"{metric}_noise_{noise:g}" for metric, noise in wide.columns]
wide = wide.reset_index()

wide

,setup,cluster_size,model_kind,partition_method,ave_auc_group_prc_noise_0,ave_auc_group_prc_noise_25,ave_auc_group_prc_noise_75,ave_auc_lf_prc_noise_0,ave_auc_lf_prc_noise_25,ave_auc_lf_prc_noise_75,ave_group_cosine_noise_0,ave_group_cosine_noise_25,ave_group_cosine_noise_75,ave_lf_cosine_noise_0,ave_lf_cosine_noise_25,ave_lf_cosine_noise_75
0,Nonlinear,50,bio,global,0.470,0.328,0.159,0.573,0.385,0.181,0.598,0.621,0.782,0.472,0.549,0.766
1,Nonlinear,50,bio,oracle,0.624,0.336,0.163,0.514,0.247,0.119,0.416,0.612,0.771,0.525,0.725,0.833
2,Nonlinear,50,bio,phate,0.406,0.270,0.210,0.350,0.219,0.152,0.620,0.657,0.739,0.669,0.744,0.800
3,Nonlinear,50,bio,random,0.383,0.290,0.101,0.316,0.225,0.081,0.666,0.681,0.814,0.746,0.760,0.862
4,Nonlinear,50,bio,raw_kmeans,0.408,0.290,0.167,0.357,0.239,0.117,0.590,0.653,0.761,0.659,0.721,0.838
5,Nonlinear,50,linear,global,0.448,0.266,0.157,0.538,0.298,0.178,0.548,0.678,0.769,0.464,0.646,0.748
6,Nonlinear,50,linear,oracle,0.545,0.255,0.252,0.425,0.213,0.174,0.498,0.691,0.678,0.636,0.772,0.770
7,Nonlinear,50,linear,phate,0.426,0.234,0.136,0.349,0.198,0.109,0.544,0.688,0.769,0.658,0.756,0.828
8,Nonlinear,50,linear,random,0.422,0.248,0.155,0.350,0.202,0.118,0.557,0.656,0.763,0.665,0.754,0.825
9,Nonlinear,50,linear,raw_kmeans,0.491,0.240,0.122,0.393,0.207,0.098,0.503,0.665,0.794,0.637,0.733,0.839


In [37]:
fixed_factors = (
    df
    .query("dataset == 'nonlinear'")
    .query("n_hidden == 5")
    .query("partition_method in ['global', 'oracle', 'phate', 'raw_kmeans', 'random']")
    .pivot_table(
        index=["setup", "cluster_size", "model_kind", "partition_method"],
        columns="n_noise",
        values=metrics,
        aggfunc="mean",
    )
    .round(3)
 )

fixed_factors.columns = [f"{metric}_noise_{noise:g}" for metric, noise in fixed_factors.columns]
fixed_factors = fixed_factors.reset_index()
fixed_factors

,setup,cluster_size,model_kind,partition_method,ave_auc_group_prc_noise_0,ave_auc_group_prc_noise_25,ave_auc_group_prc_noise_75,ave_auc_lf_prc_noise_0,ave_auc_lf_prc_noise_25,ave_auc_lf_prc_noise_75,ave_group_cosine_noise_0,ave_group_cosine_noise_25,ave_group_cosine_noise_75,ave_lf_cosine_noise_0,ave_lf_cosine_noise_25,ave_lf_cosine_noise_75
0,Nonlinear,50,bio,global,0.470,0.328,0.159,0.573,0.385,0.181,0.598,0.621,0.782,0.472,0.549,0.766
1,Nonlinear,50,bio,oracle,0.624,0.336,0.163,0.514,0.247,0.119,0.416,0.612,0.771,0.525,0.725,0.833
2,Nonlinear,50,bio,phate,0.406,0.270,0.210,0.350,0.219,0.152,0.620,0.657,0.739,0.669,0.744,0.800
3,Nonlinear,50,bio,random,0.383,0.290,0.101,0.316,0.225,0.081,0.666,0.681,0.814,0.746,0.760,0.862
4,Nonlinear,50,bio,raw_kmeans,0.408,0.290,0.167,0.357,0.239,0.117,0.590,0.653,0.761,0.659,0.721,0.838
5,Nonlinear,50,linear,global,0.448,0.266,0.157,0.538,0.298,0.178,0.548,0.678,0.769,0.464,0.646,0.748
6,Nonlinear,50,linear,oracle,0.545,0.255,0.252,0.425,0.213,0.174,0.498,0.691,0.678,0.636,0.772,0.770
7,Nonlinear,50,linear,phate,0.426,0.234,0.136,0.349,0.198,0.109,0.544,0.688,0.769,0.658,0.756,0.828
8,Nonlinear,50,linear,random,0.422,0.248,0.155,0.350,0.202,0.118,0.557,0.656,0.763,0.665,0.754,0.825
9,Nonlinear,50,linear,raw_kmeans,0.491,0.240,0.122,0.393,0.207,0.098,0.503,0.665,0.794,0.637,0.733,0.839


In [38]:
partition_compare = (
    df
    .query("dataset == 'nonlinear'")
    .query("model_kind == 'linear'")
    .query("n_hidden == 5")
    .groupby(["setup", "cluster_size", "n_noise", "partition_method"])[
        [
            "ave_group_cosine",
            "ave_auc_group_prc",
            "ave_lf_cosine",
            "ave_auc_lf_prc",
            "partition_ari",
            "cluster_purity",
            "corex_time",
            "partition_time",
            "total_time",
        ]
    ]
    .mean()
    .round(3)
 )

partition_compare

ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc  partition_ari  cluster_purity  corex_time  partition_time  total_time
setup     cluster_size n_noise partition_method                                                                                                                                           
Nonlinear 50           0       global                       0.548              0.448          0.464           0.538            NaN             NaN       0.082             NaN       0.082
                               oracle                       0.498              0.545          0.636           0.425          1.000           1.000       0.184           0.000       0.184
                               phate                        0.544              0.426          0.658           0.349          0.001           0.473       0.153           0.191       0.344
                               random                       0.557              0.422          0.665           0.350          0.005           0.552       0.127           0.000       0.127
                               raw_kmeans                   0.503              0.491          0.637           0.393          0.003           0.512       0.134           0.012       0.146
                       25      global                       0.678              0.266          0.646           0.298            NaN             NaN       0.121             NaN       0.121
                               oracle                       0.691              0.255          0.772           0.213          1.000           1.000       0.216           0.000       0.216
                               phate                        0.688              0.234          0.756           0.198         -0.001           0.535       0.181           0.140       0.321
                               random                       0.656              0.248          0.754           0.202          0.005           0.552       0.182           0.000       0.182
                               raw_kmeans                   0.665              0.240          0.733           0.207         -0.004           0.518       0.175           0.012       0.187
                       75      global                       0.769              0.157          0.748           0.178            NaN             NaN       0.110             NaN       0.110
                               oracle                       0.678              0.252          0.770           0.174          1.000           1.000       0.231           0.000       0.231
                               phate                        0.769              0.136          0.828           0.109         -0.007           0.490       0.269           0.183       0.453
                               random                       0.763              0.155          0.825           0.118          0.005           0.552       0.190           0.000       0.190
                               raw_kmeans                   0.794              0.122          0.839           0.098         -0.002           0.492       0.212           0.017       0.229
          500          0       global                       0.548              0.598          0.433           0.761            NaN             NaN       0.078             NaN       0.078
                               oracle                       0.513              0.678          0.564           0.550          1.000           1.000       0.104           0.000       0.104
                               phate                        0.534              0.529          0.662           0.405         -0.000           0.506       0.113           2.137       2.249
                               random                       0.521              0.525          0.636           0.421          0.000           0.511       0.101           0.000       0.101
                               raw_kmeans                   0.555              0.508          0.664           0.442       

In [39]:
time_summary = (
    df
    .query("dataset == 'nonlinear'")
    .groupby(["setup", "cluster_size", "model_kind", "partition_method", "n_noise"])[
        ["corex_time", "partition_time", "total_time"]
    ]
    .mean()
    .round(3)
 )

time_summary

corex_time  partition_time  total_time
setup     cluster_size model_kind partition_method n_noise                                        
Nonlinear 50           bio        global           0             1.845             NaN       1.845
                                                   25            4.054             NaN       4.054
                                                   75            7.828             NaN       7.828
                                  oracle           0             2.273           0.000       2.273
                                                   25            5.543           0.000       5.543
                                                   75           11.327           0.000      11.327
                                  phate            0             3.117           0.191       3.308
                                                   25            5.797           0.140       5.937
                                                   75            9.316           0.183       9.499
                                  random           0             2.711           0.000       2.711
                                                   25            6.073           0.000       6.073
                                                   75           10.662           0.000      10.662
                                  raw_kmeans       0             2.962           0.012       2.975
                                                   25            5.213           0.012       5.225
                                                   75           10.885           0.017      10.903
                       linear     global           0             0.082             NaN       0.082
                                                   25            0.121             NaN       0.121
                                                   75            0.110             NaN       0.110
                                  oracle           0             0.184           0.000       0.184
                                                   25            0.216           0.000       0.216
                                                   75            0.231           0.000       0.231
                                  phate            0             0.153           0.191       0.344
                                                   25            0.181           0.140       0.321
                                                   75            0.269           0.183       0.453
                                  random           0             0.127           0.000       0.127
                                                   25            0.182           0.000       0.182
                                                   75            0.190           0.000       0.190
                                  raw_kmeans       0             0.134           0.012       0.146
                                                   25            0.175           0.012       0.187
                                                   75            0.212           0.017       0.229
          500          bio        global           0            11.043             NaN      11.043
                                                   25           22.128             NaN      22.128
                                                   75           47.912             NaN      47.912
                                  oracle           0            11.346           0.000      11.346
                                                   25           23.836           0.000      23.836
                                                   75           47.924           0.000      47.924
                                  phate            0            10.633           2.137      12.770
                                                   25           22.110           2.142      24.252
                                                   75           45.182           2.193      47.375
 

# Nonlinear Timing Views

In [40]:
time_pivot = (
    df
    .query("dataset == 'nonlinear'")
    .pivot_table(
        index=["cluster_size", "model_kind", "partition_method"],
        columns="n_noise",
        values=["corex_time", "partition_time", "total_time"],
        aggfunc="mean",
    )
    .round(3)
 )

time_pivot.columns = [f"{metric}_noise_{noise:g}" for metric, noise in time_pivot.columns]
time_pivot = time_pivot.reset_index()

time_pivot

,cluster_size,model_kind,partition_method,corex_time_noise_0,corex_time_noise_25,corex_time_noise_75,partition_time_noise_0,partition_time_noise_25,partition_time_noise_75,total_time_noise_0,total_time_noise_25,total_time_noise_75
0,50,bio,global,1.845,4.054,7.828,NaN,NaN,NaN,1.845,4.054,7.828
1,50,bio,oracle,2.273,5.543,11.327,0.000,0.000,0.000,2.273,5.543,11.327
2,50,bio,phate,3.117,5.797,9.316,0.191,0.140,0.183,3.308,5.937,9.499
3,50,bio,random,2.711,6.073,10.662,0.000,0.000,0.000,2.711,6.073,10.662
4,50,bio,raw_kmeans,2.962,5.213,10.885,0.012,0.012,0.017,2.975,5.225,10.903
5,50,linear,global,0.082,0.121,0.110,NaN,NaN,NaN,0.082,0.121,0.110
6,50,linear,oracle,0.184,0.216,0.231,0.000,0.000,0.000,0.184,0.216,0.231
7,50,linear,phate,0.153,0.181,0.269,0.191,0.140,0.183,0.344,0.321,0.453
8,50,linear,random,0.127,0.182,0.190,0.000,0.000,0.000,0.127,0.182,0.190
9,50,linear,raw_kmeans,0.134,0.175,0.212,0.012,0.012,0.017,0.146,0.187,0.229


In [41]:
df

,dataset,rep,cluster_size,n_noise,partition_method,model_kind,n_hidden,corex_time,total_time,ave_group_cosine,ave_auc_group_prc,ave_lf_cosine,ave_auc_lf_prc,partition_ari,partition_time,cluster_purity,setup
0,nonlinear,0,50,0,global,linear,5,0.116054,0.116054,0.527124,0.503530,0.418923,0.629263,NaN,NaN,NaN,Nonlinear
1,nonlinear,0,50,0,global,bio,5,2.207100,2.207100,0.644099,0.484028,0.447847,0.585714,NaN,NaN,NaN,Nonlinear
2,nonlinear,0,50,0,oracle,linear,5,0.147953,0.147969,0.488211,0.575227,0.708765,0.435097,1.000000,0.000017,1.0000,Nonlinear
3,nonlinear,0,50,0,oracle,bio,5,2.516576,2.516593,0.447703,0.588542,0.584875,0.465923,1.000000,0.000017,1.0000,Nonlinear
4,nonlinear,0,50,0,phate,linear,5,0.154249,0.369316,0.517655,0.402759,0.663467,0.321427,0.031485,0.215066,0.4000,Nonlinear
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,nonlinear,3,5000,75,phate,bio,5,388.310904,397.017854,0.821645,0.201413,0.888428,0.141882,0.000580,8.706950,0.4873,Nonlinear
356,nonlinear,3,5000,75,raw_kmeans,linear,5,0.472087,0.528728,0.626818,0.556297,0.752857,0.359840,0.000695,0.056641,0.5141,Nonlinear
357,nonlinear,3,5000,75,raw_kmeans,bio,5,476.189977,476.246618,0.628372,0.524744,0.794414,0.335963,0.000695,0.056641,0.5141,Nonlinear
358,nonlinear,3,5000,75,random,linear,5,0.561286,0.561451,0.625539,0.570928,0.739759,0.393929,-0.000039,0.000165,0.5039,Nonlinear


In [42]:
summary = (
    df
    .query("dataset == 'nonlinear'")
    .groupby(["setup", "cluster_size", "model_kind", "partition_method"])[metrics]
    .mean(numeric_only=True)
    .round(3)
 )

summary = (
    summary
    .reset_index()
    .sort_values(
        ["setup", "cluster_size", "model_kind", "ave_auc_group_prc"],
        ascending=[True, True, True, False],
    )
    .set_index(["setup", "cluster_size", "model_kind", "partition_method"])
)

summary

ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc
setup     cluster_size model_kind partition_method                                                                    
Nonlinear 50           bio        oracle                       0.600              0.375          0.694           0.293
                                  global                       0.667              0.319          0.595           0.380
                                  phate                        0.672              0.295          0.738           0.241
                                  raw_kmeans                   0.668              0.288          0.739           0.238
                                  random                       0.720              0.258          0.789           0.207
                       linear     oracle                       0.623              0.350          0.726           0.271
                                  global                       0.665              0.290          0.619           0.338
                                  raw_kmeans                   0.654              0.284          0.736           0.232
                                  random                       0.658              0.275          0.748           0.224
                                  phate                        0.667              0.265          0.747           0.219
          500          bio        oracle                       0.423              0.669          0.558           0.483
                                  global                       0.625              0.457          0.543           0.532
                                  random                       0.657              0.386          0.743           0.297
                                  phate                        0.683              0.333          0.775           0.259
                                  raw_kmeans                   0.692              0.330          0.752           0.287
                       linear     oracle                       0.559              0.620          0.636           0.456
                                  global                       0.611              0.456          0.527           0.568
                                  random                       0.642              0.344          0.725           0.267
                                  raw_kmeans                   0.659              0.322          0.744           0.262
                                  phate                        0.642              0.320          0.726           0.247
          5000         bio        oracle                       0.466              0.632          0.651           0.425
                                  random                       0.597              0.586          0.752           0.394
                                  global                       0.629              0.526          0.551           0.619
                                  raw_kmeans                   0.614              0.523          0.711           0.410
                                  phate                        0.670              0.452          0.791           0.315
                       linear     oracle                       0.541              0.623          0.571           0.527
                                  random                       0.622              0.574          0.726           0.418
                                  global                       0.651              0.564          0.490           0.742
                                  raw_kmeans                   0.680              0.563          0.724           0.475
                                  phate                        0.620              0.499          0.767           0.338